# 03 · Construir el subconjunto curado con percentiles

Práctica basada en `notebooks/01_viviendas.ipynb`. Tiempo sugerido: 20–30 minutos.

**Tu tarea:** completá únicamente la función de la sección 3.
La carga de datos, el entorno y las pruebas ya están preparados.

1. Elegí el kernel del entorno `.venv` del proyecto y ejecutá las celdas de preparación.
2. Leé el contrato y reemplazá el `raise NotImplementedError` por tu implementación.
3. Ejecutá de nuevo la celda de la función y después la de pruebas.
4. Si todas pasan, esa misma celda probará tu función sobre los datos reales y mostrará el resultado.

Podés usar **Ejecutar todo** desde un kernel nuevo. Mientras la función esté pendiente,
la última celda mostrará un aviso; no tenés que instalar nada adicional en el entorno actual.

## 1. Preparación lista

La base equivale a `df` del notebook original: dormitorios imputados, marca de imputación y tres ratios. No se aplicó el filtro de curado.

Los archivos de `../datos/` ya están descargados. Si faltan, se descargan una vez desde Kaggle y se guardan localmente.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from pandas.testing import assert_frame_equal

# Encuentra la carpeta tanto desde la raíz del proyecto como desde este notebook.
ubicaciones = [Path.cwd(), *Path.cwd().parents]
carpeta = next(
    (
        candidata
        for ubicacion in ubicaciones
        for candidata in [ubicacion, ubicacion / "practicas_livecoding"]
        if candidata.name == "practicas_livecoding"
        and (candidata / "preparacion.py").is_file()
    ),
    None,
)
if carpeta is None:
    raise FileNotFoundError("Abrí el notebook dentro de la carpeta practicas_livecoding.")

if str(carpeta) not in sys.path:
    sys.path.insert(0, str(carpeta))

from preparacion import cargar_viviendas, cargar_energia

pd.set_option("display.max_columns", 20)

In [2]:
datos = cargar_viviendas()

print(f"Filas: {len(datos):,} | Columnas: {len(datos.columns)}")
display(datos.head())

Filas: 20,640 | Columnas: 14


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,bedrooms_imputados,rooms_per_household,bedrooms_per_room,population_per_household
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,False,6.984127,0.146591,2.555556
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,False,6.238137,0.155797,2.109842
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,False,8.288136,0.129516,2.802260
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,False,5.817352,0.184458,2.547945
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,False,6.281853,0.172096,2.181467


## 2. Consigna y resultado esperado

Devolvé las filas que cumplan **todas** estas condiciones:

1. `median_house_value` es estrictamente menor que el máximo de la entrada.
2. `rooms_per_household`, `bedrooms_per_room` y `population_per_household`
   son menores o iguales a sus respectivos percentiles.

Calculá cada corte sobre **toda la entrada, antes de filtrar**, con
`quantile(percentil)` y su interpolación predeterminada. El valor predeterminado corresponde al P99.
Conservá columnas, índices y orden original; no agregues columnas ni modifiques la entrada.
Una entrada vacía devuelve una copia vacía. Podés asumir `0 <= percentil <= 1`.

**Atención:** usar el percentil de un solo ratio, filtrar con OR o recalcular
los cortes después de excluir el precio máximo produce otro resultado.

## 3. Tu función

Editá solo esta celda. Mantené el nombre de la función y sus parámetros para que funcionen las pruebas.

In [3]:
def curar_viviendas(datos, percentil=0.99):
    """Implementá el contrato de la sección 2 sin modificar datos."""
    # TODO: escribí tu solución acá.
    raise NotImplementedError("Completá esta función y volvé a ejecutar las pruebas.")

## 4. Pruebas y ejecución real

Un `AssertionError` indica qué resultado difiere del esperado. Primero revisá el caso pequeño que falló. Después de editar la función, ejecutá su celda antes de volver a correr estas pruebas.

Las pruebas verifican ejemplos pequeños y propiedades del resultado real; no sustituyen una revisión de tu razonamiento.

In [4]:
# Estas pruebas no reemplazan tu función: comprueban su resultado.
def comparar(actual, esperado):
    assert isinstance(actual, pd.DataFrame), "La función debe devolver un DataFrame."
    assert_frame_equal(
        actual, esperado,
        check_dtype=False, check_index_type=False, check_column_type=False,
        check_freq=False, rtol=1e-6, atol=1e-8,
    )


def probar(funcion):
    caso = pd.DataFrame({
        "median_house_value": [10, 20, 30, 40, 50, 60, 100],
        "rooms_per_household": [1, 2, 3, 4, 5, 6, 7],
        "bedrooms_per_room": [7, 6, 5, 4, 3, 2, 1],
        "population_per_household": [1, 1, 1, 1, 1, 9, 1],
        "dato_extra": list("abcdefg"),
    }, index=[9, 8, 7, 6, 5, 4, 3])
    original = caso.copy(deep=True)

    comparar(funcion(caso, percentil=0.75), caso.iloc[[2, 3, 4]])
    comparar(funcion(caso, percentil=0.5), caso.iloc[[3]])
    comparar(funcion(caso, percentil=1), caso.iloc[:6])
    comparar(funcion(caso), caso.iloc[[1, 2, 3, 4]])

    todos_en_maximo = caso.assign(median_house_value=100)
    comparar(funcion(todos_en_maximo), todos_en_maximo.iloc[:0])
    comparar(funcion(caso.iloc[:0]), caso.iloc[:0])
    comparar(caso, original)
    print("OK: tres ratios, percentiles sobre la entrada completa, máximo estricto y orden original.")


resultado = None
try:
    probar(curar_viviendas)
except NotImplementedError:
    print("PENDIENTE: completá la función de la sección 3 y volvé a ejecutar esta celda.")
else:
    # Comprobamos también que el dataset real permanezca intacto.
    antes = datos.copy(deep=True)
    resultado = curar_viviendas(datos)
    comparar(datos, antes)
    ratios = ["rooms_per_household", "bedrooms_per_room", "population_per_household"]
    assert list(resultado.columns) == list(datos.columns)
    assert resultado["median_house_value"].lt(datos["median_house_value"].max()).all()
    assert resultado[ratios].le(datos[ratios].quantile(0.99)).all().all()
    assert resultado.index.isin(datos.index).all()
    assert 0 < len(resultado) < len(datos)
    print(f"OK: dataset real. Resultado: {len(resultado):,} filas.")
    display(resultado.head(20))

PENDIENTE: completá la función de la sección 3 y volvé a ejecutar esta celda.


## Pista opcional

<details>
<summary>Abrir solo si te trabás</summary>

`quantile`, `le`, `all(axis=1)` y una máscara booleana combinada con `&`.

</details>

**Para explicar en voz alta:** ¿qué filas entran al cálculo?, ¿qué representa cada fila de la salida?, ¿qué caso límite cubre cada prueba?